 %md
 # Daily Pre-Aggregation of fact_transactions for Power BI Performance

 Reads the Gold `fact_transactions` Delta table, aggregates at daily grain + dimension foreign keys,
 and writes as a new Delta table `fact_transactions_daily_agg`.

 **Problem**: Power BI → AAS → Synapse DirectQuery scans ~896M rows per visual query,
 causing 15–65 second render times per visual on the Payment Transactions page.

 **Solution**: Pre-aggregate additive measures (`TransactionCount`, `AmountUSD`) at the daily
 grain grouped by all dimension foreign keys. DAX measures in AAS (`Pmt_Approval$%`, `_AA`, etc.)
 compute correctly over the small pre-agg table because they use `SUM`/`DIVIDE` over additive columns.

 **Expected reduction**: 896M rows → ~50K–200K rows (99.97%+ reduction).
 Estimated page load improvement: 65s → ~2–5s.

 **Schedule**: Run after Gold-FactTransactions completes.

 **Companion doc**: `pdp-dev-mcp/docs/powerbi-query-optimization-analysis.md`

In [0]:
# Import modules
import os, sys
notebook_path = dbutils.entry_point.getDbutils().notebook().getContext().notebookPath().get()
sys.path.append(f"/Workspace{os.sep.join(notebook_path.partition('notebooks')[:2])}")

from delta.tables import *
from pyspark.sql.functions import *
from Common.Environment import *
from Common.DataLakeURIs import *
from Common.DeltaTableHelper import *
from Common.Telemetry import *
from PaymentTransactions.PaymentTransactions_TableNames import *

##  Configuration

In [0]:
# Widgets for parameterization (can be set from job parameters or interactively)
dbutils.widgets.text("source_table", "hive_metastore.gold.fact_transactions", "Source Delta table")
dbutils.widgets.text("target_table", "hive_metastore.gold.fact_transactions_daily_agg", "Target agg table")
dbutils.widgets.text("target_path", "", "Target Delta path (optional, uses managed table if blank)")

SOURCE_TABLE = dbutils.widgets.get("source_table")
TARGET_TABLE = dbutils.widgets.get("target_table")
TARGET_PATH = dbutils.widgets.get("target_path") or None

print(f"Source table: {SOURCE_TABLE}")
print(f"Target table: {TARGET_TABLE}")
if TARGET_PATH:
    print(f"Target path:  {TARGET_PATH}")
else:
    print(f"Target path:  (managed table location)")

## Read source fact_transactions

In [0]:
from pyspark.sql import functions as F
from datetime import datetime

df_fact = spark.table(SOURCE_TABLE)

source_count = df_fact.count()
print(f"Source row count: {source_count:,}")

#  Define aggregation 
 
  **Dimension keys** (GROUP BY): All columns that are foreign keys to dimension tables.
  These must be preserved so AAS slicer filters continue to work.
 
  **Degenerate dimensions** (DROPPED): Transaction-level IDs that make each row unique.
  Dropping these is what enables aggregation compression.
 
  **Measures** (SUM): Additive columns that roll up correctly via SUM.


In [0]:
# --- Dimension foreign keys: keep in GROUP BY ---
DIMENSION_KEYS = [
    "Date",                # Daily grain key (joins to DimDate)
    "FirstAttemptDate",    # First attempt date (used by _FA measures)
    "PaymentMethodId",     # FK → DimPaymentMethod
    "DunningByCycleId",    # FK → DimDunningByCycle
    "BinId",               # FK → DimBin
    "AuthenticationId",    # FK → DimAuthentication
    "BillingId",           # FK → DimBilling
    "GeoId",               # FK → DimGeo
    "MerchantId",          # FK → DimMerchant
    "PurchaseId",          # FK → DimPurchase
    "ProductId",           # FK → DimProduct
    "NetworkTokenId",      # FK → DimNetworkToken
    "ChargebackId",        # FK → DimChargeback
    "CoBrandedId",         # FK → DimCoBranded
    "ResponseCodeId",      # FK → DimResponseCode (CRITICAL for approval/decline filtering)
    "TrustedMIDId",        # FK → DimTrustedMID
    "AccountUpdaterId",    # FK → DimAccountUpdater
    "OriginalPaymentDate", # Links retries to original payment date
]

# --- Degenerate dimensions: dropped to enable aggregation ---
# These are transaction-level IDs that make each row unique.
# Dropping them allows rows with the same dimension key combination to merge.
DEGENERATE_DIMENSIONS = [
    "PaymentId",           # Unique per payment transaction
    "PaymentExtendedId",   # Unique per extended payment record
    "RetryId",             # Unique per retry attempt
]

# --- Additive measures: SUM during aggregation ---
MEASURE_COLUMNS = [
    "TransactionCount",    # bigint — SUM gives total transactions
    "AmountUSD",           # decimal(38,9) — SUM gives total dollar amount
]

# --- Metadata columns: dropped (not needed for analytics) ---
METADATA_COLUMNS = [
    "IngestionTimestamp",  # Pipeline metadata, not used in DAX
]

# --- Partition column: derived from Date after aggregation ---
PARTITION_COLUMN = "YearMonth"

print(f"Dimension keys (GROUP BY): {len(DIMENSION_KEYS)} columns")
print(f"Measures (SUM):            {len(MEASURE_COLUMNS)} columns")
print(f"Degenerate dims (DROP):    {len(DEGENERATE_DIMENSIONS)} columns")
print(f"Metadata (DROP):           {len(METADATA_COLUMNS)} columns")
print(f"Partition:                 {PARTITION_COLUMN}")

# Perform daily aggregation

In [0]:
start_time = datetime.now()

# Build aggregation expressions
agg_exprs = [F.sum(F.col(c)).alias(c) for c in MEASURE_COLUMNS]
# Also count source rows per group for verification
agg_exprs.append(F.count("*").alias("_source_row_count"))

# Perform the aggregation
df_agg = (
    df_fact
    .groupBy(*DIMENSION_KEYS)
    .agg(*agg_exprs)
    # Re-derive YearMonth from Date for partitioning
    .withColumn(PARTITION_COLUMN,
                (F.year("Date") * 100 + F.month("Date")).cast("int"))
)

# Cache for multiple downstream operations (count, write, verify)
df_agg.cache()

agg_count = df_agg.count()
agg_elapsed = datetime.now() - start_time

compression_ratio = source_count / agg_count if agg_count > 0 else 0

print(f"Aggregation complete in {agg_elapsed}")
print(f"Source rows:     {source_count:,}")
print(f"Aggregated rows: {agg_count:,}")
print(f"Compression:     {compression_ratio:.1f}x ({(1 - agg_count/source_count)*100:.2f}% reduction)")

# Verify measure totals match
 Ensures SUM(TransactionCount) and SUM(AmountUSD) are identical before and after aggregation.
This is the mathematically critical invariant for correctness.


In [0]:
# Compare totals: source vs aggregated
source_totals = df_fact.select(
    *[F.sum(c).alias(f"source_{c}") for c in MEASURE_COLUMNS]
).collect()[0]

agg_totals = df_agg.select(
    *[F.sum(c).alias(f"agg_{c}") for c in MEASURE_COLUMNS]
).collect()[0]

all_match = True
for c in MEASURE_COLUMNS:
    src_val = source_totals[f"source_{c}"]
    agg_val = agg_totals[f"agg_{c}"]
    match = src_val == agg_val
    status = "✓ MATCH" if match else "✗ MISMATCH"
    print(f"  {c}: source={src_val:,}  agg={agg_val:,}  {status}")
    if not match:
        all_match = False

# Also verify _source_row_count sums to original count
computed_source_count = df_agg.select(F.sum("_source_row_count")).collect()[0][0]
count_match = computed_source_count == source_count
print(f"  _source_row_count sum: {computed_source_count:,}  expected: {source_count:,}  {'✓' if count_match else '✗'}")

if all_match and count_match:
    print("\n✓ All measure totals verified — aggregation is mathematically correct.")
else:
    raise ValueError("MISMATCH detected! Aggregation produced incorrect totals. Aborting write.")

# Write aggregated Delta table

In [0]:
write_start = datetime.now()

# Drop the internal verification column before writing
df_final = df_agg.drop("_source_row_count")

# Write as Delta, partitioned by YearMonth (same as source)
writer = (
    df_final
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy(PARTITION_COLUMN)
    .option("overwriteSchema", "true")
)

if TARGET_PATH:
    writer.save(TARGET_PATH)
    print(f"Written to path: {TARGET_PATH}")
else:
    writer.saveAsTable(TARGET_TABLE)
    print(f"Written as managed table: {TARGET_TABLE}")

write_elapsed = datetime.now() - write_start
print(f"Write completed in {write_elapsed}")

# Post-write verification

In [0]:
# Read back and verify
if TARGET_PATH:
    df_written = spark.read.format("delta").load(TARGET_PATH)
else:
    df_written = spark.table(TARGET_TABLE)

written_count = df_written.count()
print(f"Written row count: {written_count:,}")
print(f"Expected:          {agg_count:,}")
print(f"Match:             {'✓ YES' if written_count == agg_count else '✗ NO'}")

# Show partition distribution
print("\nRows per YearMonth partition:")
(
    df_written
    .groupBy(PARTITION_COLUMN)
    .count()
    .orderBy(PARTITION_COLUMN)
    .show(100, truncate=False)
)

##  Print Synapse view SQL
After running this notebook, update the Synapse Serverless view to point to the aggregated table.
The view uses NULL placeholders for dropped degenerate dimension columns so AAS schema doesn't break.

In [0]:
# Generate the Synapse view SQL for reference
# The user should run this in Synapse Serverless SQL to redirect AAS DirectQuery

# Build column list: dimension keys + measures from the agg table,
# plus NULL placeholders for degenerate dimensions
agg_columns = DIMENSION_KEYS + MEASURE_COLUMNS + [PARTITION_COLUMN]

select_parts = []
for col_name in agg_columns:
    select_parts.append(f"    r.[{col_name}]")

# Add NULL placeholders for degenerate dimensions (preserves AAS schema)
for col_name in DEGENERATE_DIMENSIONS:
    select_parts.append(f"    CAST(NULL AS BIGINT) AS [{col_name}]")

# Add NULL for IngestionTimestamp
select_parts.append(f"    CAST(NULL AS DATETIME2) AS [IngestionTimestamp]")

select_clause = ",\n".join(select_parts)

# Determine the BULK path (user should update with actual storage path)
if TARGET_PATH:
    bulk_path = TARGET_PATH.replace("abfss://", "").split("@")[1] if "@" in (TARGET_PATH or "") else "<YOUR_DELTA_PATH>"
else:
    bulk_path = "gold/fact_transactions_daily_agg/"

synapse_sql = f"""-- ================================================================
-- Synapse Serverless View: Redirect AAS to daily pre-aggregated table
-- Run this in Synapse Serverless SQL (database: aas)
-- ================================================================

CREATE OR ALTER VIEW [dbo].[fact_transactions_v_daily_agg]
AS
SELECT
{select_clause}
FROM OPENROWSET(
    BULK 'gold/fact_transactions_daily_agg/',
    DATA_SOURCE = 'DeltaLakeStorage',
    FORMAT = 'DELTA'
) AS r;

-- ================================================================
-- To switch AAS to use the aggregated view:
-- 1. Update the AAS partition M-expression / connection string
--    to reference fact_transactions_v_daily_agg instead of
--    the raw fact_transactions view.
-- 2. Or: ALTER the existing fact_transactions view to read from
--    the agg table directly (simpler, affects all partitions at once):
--
-- CREATE OR ALTER VIEW [dbo].[fact_transactions]
-- AS
-- SELECT ... FROM OPENROWSET(
--     BULK 'gold/fact_transactions_daily_agg/',
--     DATA_SOURCE = 'DeltaLakeStorage',
--     FORMAT = 'DELTA'
-- ) AS r;
-- ================================================================
"""

print(synapse_sql)

## Summary

In [0]:
total_elapsed = (datetime.now() - start_time)

print("=" * 70)
print("DAILY PRE-AGGREGATION SUMMARY")
print("=" * 70)
print(f"Source table:    {SOURCE_TABLE}")
print(f"Target table:    {TARGET_TABLE}")
print(f"Source rows:     {source_count:,}")
print(f"Aggregated rows: {agg_count:,}")
print(f"Compression:     {compression_ratio:.1f}x ({(1 - agg_count/source_count)*100:.2f}% reduction)")
print(f"Measure totals:  VERIFIED ✓")
print(f"Duration:        {total_elapsed}")
print("=" * 70)
print()
print("NEXT STEPS:")
print("  1. Run the Synapse view SQL above in Synapse Serverless SQL")
print("  2. Process the AAS model to pick up the new view")
print("  3. Open the Power BI report and run Performance Analyzer")
print("  4. Compare results against the baseline in:")
print("     pdp-dev-mcp/docs/powerbi-query-optimization-analysis.md")
print()
print("ROLLBACK:")
print("  To revert, point the Synapse view back to the raw table:")
print(f"  BULK 'gold/fact_transactions/' instead of 'gold/fact_transactions_daily_agg/'")

# Unpersist cached dataframe
df_agg.unpersist()

In [0]:
df_gold = spark.table("hive_metastore.gold.fact_transactions")
import pyspark.sql.functions as F
display(df_gold.agg(F.min("Date").alias("min_date")))


In [0]:
%sql
SELECT MIN(YearMonth) AS EarliestMonth, MAX(YearMonth) AS LatestMonth,
       COUNT(*) AS TotalRows, COUNT(DISTINCT YearMonth) AS DistinctMonths
FROM gold.fact_transactions